# 📊 Part B: Data Analysis & Testing Tasks

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Load Dataset

In [ ]:
df = pd.read_csv('spread_locator_dataset.csv')
print(df.shape)
df.head()


(220, 7)


,transaction_id,customer_id,transaction_amount,transaction_date,transaction_count,region,transaction_status
0,e98aa092-3770-4fdb-9502-5b5a6a244811,CUST2824,3821.34,2023-01-26,3,North,Fail
1,11ba6918-dba0-41e5-96cf-f5a7b95f0103,CUST1409,2781.84,2023-01-28,0,East,Fail
2,82b7654b-6eb7-4579-89a0-1a9edec0a7bb,CUST5506,4120.97,2023-01-28,0,South,Fail
3,f7166574-f400-4d53-b526-0b11f6619ddf,CUST5012,6383.78,2023-01-18,2,South,Success
4,8632fe26-b507-4068-9c68-1b2fa04fecb3,CUST4657,2651.61,2023-01-04,4,North,Success


## Extract Transaction Amount

In [ ]:
amounts = df['transaction_amount']
per_day = df['transaction_count']

# Bernoulli variable
occurred = (df['transaction_status'] == 'Success').astype(int)
print(occurred.value_counts())

transaction_status
0    1109
1     891
Name: count, dtype: int64


## 1. Bernoulli

In [ ]:
p = occurred.mean()
print("Probability of success:", round(p, 4))

Probability of success: 0.4455


## 2. Binomial

In [ ]:
np.random.seed(42)
n = 10
binomial = np.random.binomial(n, p, 1000)
print("First 10 samples:", binomial[:10])
print("Simulated mean:", round(binomial.mean(), 3), "| Expected (n*p):", round(n*p, 3))

First 10 samples: [4 7 5 5 3 3 2 6 5 5]
Simulated mean: 4.414 | Expected (n*p): 4.455


## 3. Poisson

In [ ]:
lam = per_day.mean()
poisson = np.random.poisson(lam, 1000)
print("Lambda (avg transactions/day):", round(lam, 4))
print("First 10 samples:", poisson[:10])

Lambda (avg transactions/day): 2.8545
First 10 samples: [4 4 3 5 8 1 2 1 2 3]


## 4. Log-Normal

In [ ]:
shape, loc, scale = stats.lognorm.fit(amounts)
print(f"Shape (sigma): {shape:.4f}")
print(f"Loc (shift):   {loc:.4f}")
print(f"Scale (exp(mu)): {scale:.4f}")
print(f"Fitted mu (log-space mean): {np.log(scale):.4f}")

Shape (sigma): 0.5417
Loc (shift):   333.7723
Scale (exp(mu)): 2604.7407
Fitted mu (log-space mean): 7.8656


## 5. Power Law

In [ ]:
a, loc_pw, scale_pw = stats.powerlaw.fit(amounts)
print(f"Shape (a):  {a:.4f}")
print(f"Loc:        {loc_pw:.4f}")
print(f"Scale:      {scale_pw:.4f}")

Shape (a):  0.4123
Loc:        804.4200
Scale:      19658.4200


## 6. Q-Q Plot

In [ ]:
(osm, osr), (slope, intercept, r) = stats.probplot(amounts, dist="norm")
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(osm, osr, 'o', markersize=3, alpha=0.4, color='steelblue', label='Data quantiles')
ax.plot(osm, slope * np.array(osm) + intercept, 'r-', linewidth=2, label='Normal fit')
ax.set_title("Q-Q Plot of Transaction Amounts")
ax.set_xlabel("Theoretical Quantiles")
ax.set_ylabel("Sample Quantiles")
ax.legend()
plt.tight_layout()
plt.show()
print(f"R² (normality fit): {r**2:.4f}")

## 7. Box-Cox Transformation

In [ ]:
positive = amounts[amounts > 0]
transformed, lam_bc = stats.boxcox(positive)
print(f"Box-Cox lambda: {lam_bc:.6f}")
print(f"Original skewness:    {positive.skew():.4f}")
print(f"Transformed skewness: {pd.Series(transformed).skew():.4f}")

Box-Cox lambda: -0.180834
Original skewness:    0.5826
Transformed skewness: -0.0013


## 8. Z-Score & Probability

In [ ]:
z = stats.zscore(amounts.values)
prob = 1 - stats.norm.cdf(5000, amounts.mean(), amounts.std())
print("Z-scores (first 5):", [round(v, 4) for v in z[:5]])
print(f"Mean: {amounts.mean():.2f}, Std: {amounts.std():.2f}")
print(f"P(transaction_amount > 5000) = {prob:.6f}")

Z-scores (first 5): [0.2302, -0.2944, 0.3815, 1.5236, -0.3602]
Mean: 3279.30, Std: 1841.64
P(transaction_amount > 5000) = 0.205172


## 9. PDF & CDF

In [ ]:
x = np.linspace(amounts.min(), amounts.max(), 100)
pdf = stats.norm.pdf(x, amounts.mean(), amounts.std())
cdf = stats.norm.cdf(x, amounts.mean(), amounts.std())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(x, pdf, color='steelblue', linewidth=2)
axes[0].set_title("PDF of Transaction Amounts")
axes[0].set_xlabel("Transaction Amount")
axes[0].set_ylabel("Probability Density")

axes[1].plot(x, cdf, color='darkorange', linewidth=2)
axes[1].set_title("CDF of Transaction Amounts")
axes[1].set_xlabel("Transaction Amount")
axes[1].set_ylabel("Cumulative Probability")

plt.tight_layout()
plt.show()

# 🔚 Conclusion

In this analysis, we explored different statistical distributions using the given transaction dataset.

- The **Bernoulli distribution** showed P(success) = **0.4455** — ~44.6% of transactions succeeded.
- The **Binomial distribution** simulated 10-trial outcomes; the simulated mean (4.414) closely matched the expected value (n×p = 4.455).
- The **Poisson distribution** modelled transactions per day with λ = **2.8545**.
- The **Log-normal distribution** fit yielded σ ≈ 0.54, confirming right-skewed transaction amounts.
- The **Power law distribution** confirmed a heavy-tailed distribution in transaction amounts.
- The **Q-Q plot** showed moderate deviation from normality, especially in the tails.
- The **Box-Cox transformation** (λ = -0.18) reduced skewness from 0.58 to nearly 0, normalising the data.
- The **Z-score** identified how far each transaction deviates from the mean; P(amount > 5000) ≈ 20.5%.
- The **PDF and CDF** provided a visual understanding of the probability distribution of transaction amounts.

Overall, this analysis demonstrates how statistical methods can be applied to real-world financial data to gain meaningful insights.
